# Reference Guide

## Core models

### Modeling background and motivation

We introduce a generalized optimization problem, termed process family design, which aims to simultaneously determine:

1. **The Process Platform**: The optimal unit module designs for all shared unit module types within the platform.

2. **The Process Variants (Configuration)**: The selection of a specific unit module design for each shared unit module type across all process variants.

3. **The Process Variants (Operation)**: The optimal variant-specific designs and operating variables for all unique unit modules outside of the platform.

To solve this problem, we present a rigorous optimization approach that concurrently evaluates all three tiers using a Mixed-Integer Non-Linear Programming (MINLP) formulation embedded with Generalized Disjunctive Programming (GDP). More information on GDP can be found on [Pyomo GDP](https://pyomo.readthedocs.io/en/6.8.0/modeling_extensions/gdp/index.html#). The objective function minimizes the total annualized cost of the system for a given level of standardization. Within this formulation, each common unit module type $c \in C$ is assigned exactly one candidate platform design $\hat{d}_{c,l}$, with these discrete structural decisions explicitly modeled via logical disjunctions {cite:p}`stinchfield2024mixed`,{cite:p}`stinchfield2025mixed`.

$$
\begin{aligned}
\min_{d,o,\hat d, Y} \quad & \sum_{v\in V} w_v \, p_v &&&& \text{(1a)} \\
\text{s.t.} \quad & p_v = \phi_v(r_v,d_v,o_v), && \forall v\in V, && \text{(1b)} \\
& i_v = \psi_v(r_v,d_v,o_v), && \forall v\in V, && \text{(1c)} \\
& 0 = h_v(r_v,d_v,o_v), && \forall v\in V, && \text{(1d)} \\
& \bigvee_{l\in L_c} \left[ Y_{v,c,l} \wedge \left(d_{v,c} = \hat d_{c,l}\right) \right] && \forall v\in V,\ c\in C, && \text{(1e)} \\
& d_v^L \le d_v \le d_v^U, && \forall v\in V, && \text{(1f)} \\
& \hat d_{c,l}^L \le \hat d_{c,l} \le \hat d_{c,l}^U, && \forall c\in C,\ l\in L_c, && \text{(1g)} \\
& o_v^L \le o_v \le o_v^U, && \forall v\in V, && \text{(1h)} \\
& i_v^L \le i_v \le i_v^U, && \forall v\in V, && \text{(1i)} \\
& Y_{v,c,l} \in \{\text{True},\text{False}\}. &&&& \text{(1j)}
\end{aligned}
$$

The objective function, defined in Equation $\text{(1a)}$, minimizes the weighted total annualized cost across all process variants, where $w_v$ represents the variant-specific weight and $p_v$ is the annualized cost variable for variant $v\in V$. Constraint $\text{(1b)}$ establishes the cost equations, mapping process variant requirements $r_v$, structural design variables $d_v$, and operating variables $o_v$ to the calculated cost $p_v$. Equations $\text{(1c)}$ and $\text{(1d)}$ represent the systems of equations, also functions of these three variables, governing performance indicators $i_v$ and the equation-oriented process system model, respectively {cite:p}`stinchfield2024mixed`,{cite:p}`stinchfield2025mixed`. 

Constraint $\text{(1e)}$ defines the logical disjunctions for selecting the unit module design for each common unit module type across all process variants. Here, $d_{c,l}$ represents the design variable for common unit module type $c\in C$ under candidate design $l\in L_c$, while $d_{v,c}$ is the design variable assigned to process variant $v\in V$ for common unit module type $c\in C$. Finally, the constraints in $\text{(1g)} - \text{(1i)}$ define the variable bounds {cite:p}`stinchfield2024mixed`,{cite:p}`stinchfield2025mixed`.

Applying standard transformations to the disjunctions inherently expands the problem's combinatorial scale. When combined with the nonlinearities originating from the process models themselves, specifically in constraints $\text{(1b)} - \text{(1d)}$, the result is a large-scale, highly complex problem. This class of problem poses significant convergence and computational time challenges for standard MINLP solvers. To address this, we developed two modeling reduction approaches that significantly decrease problem size and mathematical complexity. This foundational work led to four distinct optimization-based techniques for solving process family design problems {cite:p}`stinchfield2024mixed`,{cite:p}`stinchfield2025mixed`.

1. **Discretization approach** {cite:p}`stinchfield2024mixed`
2. **MILP-representable ML Surrogates approach** {cite:p}`stinchfield2025mixed`
3. **Decomposition of the Discretization approach** {cite:p}`stinchfield2024progressive`
4. **Discretization approach with Economies of Numbers savings** {cite:p}`stinchfield2024economies`

For an in-depth discussion and rigorous validation of these methods, please refer to the four publications listed in the [Further Reading](#further-reading) section, where each numbered approach corresponds to its respective paper citation. 

The following sections summarize the core execution themes of these methodologies and connect them to the example workflows provided in this repository.

### The discretization approach

The discretization approach maps the continuous structural design space onto a finite set of candidate configurations. By predefining a discrete set of candidate designs for the common unit modules, this approach transforms a highly complex MINLP into a mixed-integer linear programming (MILP) formulation that can be solved efficiently while preserving the core trade-offs between shared platform elements and customized features {cite:p}`stinchfield2024mixed`.

The implementation operates in two sequential phases:

1. **Generation of Design Alternatives**: A large set of individual optimization problems is solved using the process system models and bounds in constraints $\text{(1b)} - \text{(1i)}$. This step maps out all mathematically feasible design alternatives $A_v$ for each variant and calculates their corresponding annualized costs $p_{v,a}$ {cite:p}`stinchfield2024mixed`.
2. **Platform Selection**: Once the feasible design space is discretized and the costs are pre-computed, the resulting MILP is solved to simultaneously achieve the three structural design goals outlined previously {cite:p}`stinchfield2024mixed`.

In this MILP formulation, the structural choices are governed by the following key variables and parameters {cite:p}`stinchfield2024mixed`:
*   $x_{v,a}$: A binary decision variable indicating whether alternative design $a \in A_v$ is selected for process variant $v \in V$.
*   $z_{c,l}$: A binary decision variable indicating whether candidate design $l \in L$ is selected to be available within the process platform for common unit module type $c \in C$.
*   $N_c$: A user-defined parameter dictating the maximum number of alternative designs permitted in the process platform for each common unit module type $c \in C$.

$$
\begin{aligned}
\min_{x,z} \quad & \sum_{v\in V}\sum_{a\in A_v} w_v p_{v,a} x_{v,a} &&&& \text{(2a)} \\
\text{s.t.} \quad & \sum_{a\in A_v} x_{v,a} = 1 && \forall v\in V, && \text{(2b)} \\
& \sum_{l\in L_c} z_{c,l} \le N_c, && \forall c\in C, && \text{(2c)} \\
& x_{v,a} \le z_{c,l}, && \forall (c,l)\in Q_a, && \text{(2d)} \\
& z_{c,l}\in\{0,1\}, \quad 0\le x_{v,a}\le 1. &&&& \text{(2e)}
\end{aligned}
$$

The objective function in Equation $\text{(2a)}$ minimizes the total annualized cost across all process variants. Constraint $\text{(2b)}$ enforces that exactly one alternative design is selected for each process variant $v \in V$. Constraint $\text{(2c)}$ bounds the platform capacity, defining the maximum number of alternative designs permitted for each common unit module type within the shared process platform {cite:p}`stinchfield2024mixed`.

To maintain structural consistency, Constraint $\text{(2d)}$ acts as a logical gate keeper, preventing the selection of any variant alternative design unless its required common unit module design has been explicitly made available in the process platform. Lastly, Equations $\text{(2e)}$ define the variable domains and bounds for all decision variables {cite:p}`stinchfield2024mixed`.

The mathematical formulation and underlying optimization algorithms for this methodology are implemented in the [`discretized.py`](process_family/type/discretized.py) file.


### The MILP-representable ML surrogates approach

The surrogate-based formulation utilizes machine learning models to approximate complex, highly nonlinear process relationships. By embedding these specialized surrogates directly into the mathematical model, the framework explores the continuous design space comprehensively while maintaining a tractable MILP or piecewise-linear structural format. The specific surrogate-modeling techniques, training strategies, and architecture choices are detailed in the [`trainer` module folder](process_family/utils/trainer) {cite:p}`stinchfield2025mixed`.

In this formulation, the original nonlinear systems of equations, originally defined in constraints $\text{(1b)} - \text{(1d)}$, are replaced by piecewise-linear machine learning surrogate predictions. This preserves the overall structural integrity of the original generalized disjunctive programming (GDP) formulation, but swaps out computationally expensive nonlinear physics equations for surrogate-based linear constraints {cite:p}`stinchfield2025mixed`.

In this MILP formulation, the structural choices are governed by the following key variables and parameters {cite:p}`stinchfield2025mixed`:
*   $f_v$ and $g_v$: The trained, piecewise-linear machine learning surrogate models mapping requirements $r_v$, design choices $d_v$, and operating variables $o_v$ to annualized costs and performance indicators, respectively.
*   $Y_{v,c,l}$: A boolean variable representing the logical state of the disjunction, determining if candidate design $l\in L_c$ is selected for common unit type $c\in C$ in variant $v\in V$.
*   $y_{v,c,l}$: The binary variable counterpart resulting from the mathematical transformation of the disjunction $Y_{v,c,l}$.
*   $\hat d_{c,l}$: The continuous design variable specifying the physical dimensions or capacities of the candidate platform design $l\in L_c$ for common unit type $c\in C$.

The compact form of this model reformulation is given by:

$$
\begin{aligned}
\min_{d,\hat d, o, y} \quad & \sum_{v\in V} w_v \, p_v &&&& \text{(3a)} \\
\text{s.t.} \quad & p_v = f_v(r_v,d_v,o_v), &&&& \text{(3b)} \\
& i_v = g_v(r_v,d_v,o_v), &&&& \text{(3c)} \\
& \bigvee_{l\in L_c} \left[ Y_{v,c,l} \wedge \left(d_{v,c} = \hat d_{c,l}\right) \right] && \forall v\in V,\ c\in C, && \text{(3d)} \\
& y_{v,c,l}\in\{0,1\}, \quad \hat d_{c,l}\in \mathbb{R}. &&&& \text{(3e)}
\end{aligned}
$$

The objective function in equation $\text{(3a)}$ minimizes the total weighted annualized cost across all process variants, matching the objective of the original MINLP formulation. Constraints $\text{(3b)}$ and $\text{(3c)}$ represent the linear, piecewise surrogate models, while Constraint $\text{(3d)}$ preserves the logical disjunctions from the original GDP formulation. Lastly, equations $\text{(3e)}$ establish the bounds and domains for the decision variables {cite:p}`stinchfield2025mixed`.

The critical advantage of this reformulation is the replacement of the highly non-convex, nonlinear systems of equations with linear machine learning surrogates. This substitution transforms the original MINLP into a highly tractable mixed-integer linear programming (MILP) framework. Furthermore, unlike discretization methods, this approach bypasses the need to solve a large array of preliminary optimization problems, significantly reducing initial computational overhead {cite:p}`stinchfield2025mixed`.

The mathematical formulation and underlying optimization algorithms for this methodology are implemented in the [`surrogates.py`](process_family/type/surrogates.py) file.


### Decomposition of the discretization approach

In industrial manufacturing, process families can scale rapidly. When modeled using the standard discretization approach, these large-scale systems require an intractable number of discrete design choices. This combinatorial explosion complicates the initial phase of mapping out design alternatives, causing computational times to balloon {cite:p}`stinchfield2024progressive`.

To address this scalability challenge, a decomposition technique based on Progressive Hedging was developed and tuned. This method decomposes the monolithic process family design problem into smaller, sub-family subproblems, enabling efficient parallel or sequential solution paths {cite:p}`stinchfield2024progressive`. For a rigorous breakdown of this decomposition algorithm, refer to the third publication listed under [Further Reading](#further-reading).

```{note}
Progressive Hedging decomposition functionality is currently unsupported in this repository.
```

### Discretization approach + consideration for economies of numbers savings

This methodology extends the standard discretization framework by incorporating manufacturing cost reductions driven by economies of numbers. By embedding these scale dynamics directly into the optimization formulation, the model explicitly accounts for the marginal cost savings achieved when manufacturing identical unit modules repeatedly. Consequently, the unit module procurement cost is dynamically discounted based on the total production volume of that specific module design across the entire process family. For an in-depth explanation of how these production curves are mathematically modeled {cite:p}`stinchfield2024economies`, see the fourth publication listed in [Further Reading](#further-reading).

The system variables, tracking indicators, and parameters for this formulation are defined below [[4]](#stinchfield2024economies):
*   $x_{v,a}$: A binary decision variable indicating whether design alternative $a \in A_v$ is selected for process variant $v \in V$.
*   $z_{c,l,n}$: A binary decision variable indicating whether candidate design $l\in L_c$ of common unit type $c\in C$ is manufactured exactly $n$ times ($z_{c,l,0} = 1$ implies the design is not selected for production).
*   $\rho$: A continuous variable representing the total accrued cost credit resulting from economies of numbers manufacturing savings across all module types and designs.
*   $w_v$: The weight parameter associated with process variant $v \in V$.
*   $p_{v,a}$: The base annualized cost of selecting design alternative $a \in A_v$ for process variant $v \in V$.
*   $M_c$: A parameter specifying the exact or maximum allowable number of common candidate designs made available in the platform for common unit module type $c\in C$.
*   $p_{c,l}$: The baseline, single-unit base price of candidate design $l\in L_c$ for common unit type $c\in C$.
*   $p_{c,l}^{(n)}$: The discounted unit price of candidate design $l\in L_c$ for common unit type $c\in C$ when manufactured at a production volume of $n$.
*   $Q_a$: The map set linking specific design alternatives $a \in A_v$ to their constituent candidate common unit designs $(c,l)$.
*   $A_{v,c,l}$: The subset of alternative designs for variant $v \in V$ that explicitly utilize candidate design $l\in L_c$ of common unit module type $c\in C$.

The compact form of this formulation with the economies of numbers extension is given by:

$$
\begin{aligned}
\min_{x,y,\rho} \quad & \sum_{v\in V}\sum_{a\in A_v} w_v p_{v,a} x_{v,a} - \rho &&&& \text{(4a)} \\
\text{s.t.} \quad & \sum_{a\in A_v} x_{v,a}=1, && \forall v\in V, && \text{(4b)} \\
& \sum_{l\in L_c} (1 - z_{c,l,0}) = M_c, && \forall c\in C, && \text{(4c)} \\
& x_{v,a} \le 1-z_{c,l,0}, && \forall (c,l)\in Q_a, && \text{(4d)} \\
& \sum_{n\in N}  z_{c,l,0} = 1, && \forall c\in C,l\in L_c, && \text{(4e)}\\
& \sum_{n\in N}  n\,z_{c,l,0} = \sum_{v\in V}\sum_{a\in A_{v,c,l}} w_v x_{v,a}, &&&& \text{(4f)} \\
& \rho = \sum_{c,l,n} n\,z_{c,l,n}\,(p_{c,l}-p_{c,l}^{(n)}), &&&& \text{(4g)} \\
& z_{c,l,n}\in\{0,1\}, \quad 0\le x_{v,a}\le 1. &&&& \text{(4h)}
\end{aligned}
$$

The objective function in equation $\text{(4a)}$ minimizes the total weighted annualized cost across all process variants, explicitly factoring in manufacturing cost reductions via the discount variable $\rho$. Constraint $\text{(4b)}$ ensures that exactly one design alternative is selected for each process variant $v \in V$. Constraint $\text{(4c)}$ defines the platform structural constraints by restricting the maximum number of common candidate designs permitted for each common unit module type $c \in C$ {cite:p}`stinchfield2024economies`. 

To maintain platform compliance, Constraint $\text{(4d)}$ prevents a variant from selecting a design alternative if its constituent common unit module designs have not been established within the shared platform. Constraints $\text{(4e)}$ and $\text{(4f)}$ enforce that the binary tracker evaluates to 1 if candidate design $l$ for common unit type $c$ is manufactured exactly $n$ times across the family, and 0 otherwise. Constraint $\text{(4g)}$ dynamically calculates the total realized financial credit from volume manufacturing and maps it to variable $\rho$. Lastly, equations $\text{(4h)}$ define the variable domains and bounds {cite:p}`stinchfield2024economies`.

This mathematical extension demonstrates how process family optimization models can directly internalize the economic benefits of physical production scale and manufacturing standardization.

```{note}
Economies of numbers functionality is currently unsupported in this repository.
```


## Developer documentation

This section of the documentation is intended for developers, and much of it is targeted at the IDAES/IDAES-MVO internal team. Hopefully many of the principles and ideas are also applicable to external contributors.

### IDAES-MVO contributor guide

#### About 
This page tries to give all the essential information needed to contribute software to the IDAES/IDAES-MVO project. It is designed to be useful to both internal and external collaborators.

#### Termenology

**API**
        Acronym for "Application Programming Interface", this is the
        set of functions used by an external program to invoke the
        functionality of a library or application. For IDAES/IDAES-MVO, it usually
        refers to Python functions and classes/methods in a Python module.
        By analogy, the APIs are to the IDAES-MVO library what a steering wheel,
        gearshift and pedals are to a car.

#### Code and other file locations

**Source code**
    The main Python package is under the `idaes-mvo/` directory.
    Sub-directories, aka subpackages, should be documented elsewhere.
    If you add a new directory in this tree, be sure to add a `__init__.py` in that directory
    so Python knows it is a subpackage with Python modules.

**Documentation**
    The documentation for the core package is under `docs`.

**Examples**
    Examples are under the `examples/` directory.

#### Developer environment

Development of IDAES-MVO will require an extra set of required package not needed by regular users.
To install those extra developer tools use the command ``pip install -r requirements-dev.txt``

#### Code style

The code style is not entirely consistent. But some general guidelines are:

* follow the `PEP8`_ style (or variants such as `Black`_)
* use `Google-style`_ docstrings on classes, methods, and functions
* format your docstrings as `reStructuredText`
* check your spelling using `crate-ci-typos`_
* add logging to your code by creating and using a global log object named
  for the module, which can be created like: ``_log = logging.getLogger(__name__)``
* take credit by adding a global author variable: ``__author__ = 'yourname'``

.. _PEP8: https://www.python.org/dev/peps/pep-0008/
.. _Black: https://github.com/python/black
.. _Google-style: https://sphinxcontrib-napoleon.readthedocs.io/en/latest/example_google.html
.. _reStructuredText: http://docutils.sourceforge.net/rst.html
.. _crate-ci-typos: https://github.com/crate-ci/typos

#### Tests

For general information about writing tests in Python, see :ref:`tst-top`.

There are three types of tests:

**Python source code**
    The Python tests are integrated into the Python source code directories.
    Every package (directory with `.py` modules and an `__init__.py` file)
    should also have a `tests/` sub-package, in which are test files. These,
    by convention are named `test_<something>.py`.

**Writing tests**
We use `pytest`_ to run our tests. The main advantage of this framework over
the built-in `unittest` that comes with Python is that almost no boilerplate
code is required. You write a function named `test_<something>()` and,
inside it, use the (pytest-modified) `assert` keyword to check that things
are correct.

Writing the Python unit tests in the `tests/` directory is,
hopefully, quite straightforward.
Here is an example (out of context) that tests a couple of 
things related to configuration in the core unit model library::

    def test_config_block():
        m = ConcreteModel()

        m.u = Unit()

        assert len(m.u. config) == 2
        assert m.u.config.dynamic == useDefault

See the existing tests for many more examples.


First, note that reStructuredText directive and indented Python code. The indentation of the
Python code is important. You have to write an entire program here, so all the
imports are necessary (unless you use the `testsetup` and `testcleanup` directives,
but honestly this isn't worth it unless you are doing a lot of tests in one file).
Then you write your Python code as usual.

**Running tests**

Running all tests is done by, at the top directory, running the command: ``pytest``.

You can run specific tests using the pytest syntax, see its documentation or ``pytest -h`` for details.

.. _pytest: https://docs.pytest.org/en/latest/

#### Documentation
The documentation is built from jupyter notebooks utilizing the python package jupyter[book] that writes MySt style documentation.
The components

1.  Configuration files
  a. myst.yml - The central configuration file used to manage and build a MyST project, book, or website. It defines global metadata like the title and authors, structures the table of contents, and configures project-wide build and export settings.
2. index.md - Landing page
3.references.bib - Contains all bibliographies of relevant journal articles.
3. Custom Jupyter Notebooks - Contains the raw markdown documentation.

#### Build and preview documentation
You can generate and preview documentation locally by following these steps:

**Step 1 - Build**
```bash
cd docs
```
Once you are in the docs directory under root type this comand in to have jupyter[book] build the documentation.
 ```bash
jupyter book build 
```
**Step 2 - Create a local server**

```bash
jupyter book init
```
It will prompt you asking if you want jupyter[book] to proceed (Y/n). Type Y in the terminal and press enter.

It will give you a local http url for the local survor. Paste that into your browser to preview the documentation. 

Alternatively you can follow these steps to build the documentation locally through MySt itself:

**Step 1 - Build**
```bash
cd docs
```
**Step 2 - Clean**

```bash
myst clean 
```
**Step 2 - Create a local server**
```bash
myst init
```
It will prompt you asking if you want myst to proceed (Y/n). Type Y in the terminal and press enter.

It will give you a local http url for the local survor. Paste that into your browser to preview the documentation. 

## Further reading

The bibliography for this documentation is maintained in the references file and includes the main research papers behind the repository. The content above is intended to connect those papers to the practical implementation in the repository and to the modeling workflows shown in the examples.
